# 05 — Priorização ICE

**Objetivo:** ranquear os sub-fluxos por prioridade de intervenção usando o framework ICE, e selecionar o #1 como alvo do experimento A/B.

**Fórmula:**

    ICE = (Impact × Confidence) / Ease

| Componente | Definição | Fonte |
|---|---|---|
| **Impact** | `n_conversas × taxa_any_pattern` | notebooks 01–04 |
| **Confidence** | % do padrão dominante sobre `any_pattern` | notebook 04 |
| **Ease** | Dificuldade da intervenção (1=trivial, 10=complexo) | julgamento explicitado abaixo |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

pd.set_option('display.float_format', '{:.1f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

PROC_DIR    = Path('../data/processed')
FIGURES_DIR = Path('../reports/figures')

sf = pd.read_csv(PROC_DIR / 'subflow_patterns_ice.csv')
print(f"Subflows carregados: {len(sf)}")
sf[['flow','subflow','n','any_pattern','impact','confidence','ease','ice_score','ice_norm']].head()

## 1. Justificativa do Ease por Subflow

O Ease é o único componente subjetivo do ICE. Documentamos o raciocínio por categoria:

In [ ]:
ease_rationale = [
    ('shopping_cart',  2, 'Adicionar 1 pergunta diagnostica no turno de abertura — mudanca de 1 linha de prompt'),
    ('credit_card',    3, 'Adicionar 1-2 perguntas (validade? digitou errado?) — pequena mudanca de fluxo'),
    ('slow_speed',     5, 'Redesenhar fluxo diagnostico multi-causa (browser, rede, device) — moderado'),
    ('search_results', 6, 'Padroes mistos; causa menos clara; requer analise adicional'),
]

print(f"{'Subflow':20s} {'Ease':5s}  Justificativa")
print('-' * 80)
for sf_name, ease, reason in ease_rationale:
    print(f"{sf_name:20s} {ease:5d}  {reason}")
print()
print("Demais subflows: Ease padrao = 5")

## 2. ICE Ranking — Top 15

In [ ]:
top15 = sf.nlargest(15, 'ice_score')[[
    'flow','subflow','n','any_pattern','impact','confidence','ease','ice_score','ice_norm'
]].reset_index(drop=True)
top15.index += 1
top15.index.name = 'rank'
print(top15.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

top10 = sf.nlargest(10, 'ice_score').sort_values('ice_norm')
bar_labels = top10['subflow'] + ' (' + top10['flow'] + ')'
colors_bar = ['#d62728' if s == 'shopping_cart' else
              '#ff7f0e' if s in ('credit_card','slow_speed') else
              '#aec7e8' for s in top10['subflow']]

bars = ax.barh(bar_labels, top10['ice_norm'], color=colors_bar, edgecolor='none')

for bar, row in zip(bars, top10.itertuples()):
    ax.text(bar.get_width() + 0.5,
            bar.get_y() + bar.get_height()/2,
            f'ICE={row.ice_norm:.0f}  I={row.impact:.0f} C={row.confidence:.0f} E={row.ease:.0f}',
            va='center', fontsize=8.5)

ax.set_xlabel('ICE Score Normalizado (0-100)')
ax.set_title('Ranking ICE — Top 10 Subflows', fontweight='bold', fontsize=13)
ax.set_xlim(0, 145)

red_patch    = mpatches.Patch(color='#d62728', label='#1 Alvo do A/B')
orange_patch = mpatches.Patch(color='#ff7f0e', label='#2-3 Candidatos futuros')
blue_patch   = mpatches.Patch(color='#aec7e8', label='Demais')
ax.legend(handles=[red_patch, orange_patch, blue_patch], loc='lower right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_ice_ranking.png', bbox_inches='tight')
plt.show()

## 3. Análise de Sensibilidade do Ease

O Ease é o único componente subjetivo. Verificamos se o #1 muda sob diferentes premissas de Ease.

In [ ]:
top4 = sf.nlargest(4, 'ice_score')[['subflow','impact','confidence']].copy()

ease_range = [1, 2, 3, 4, 5, 6, 7, 8]
print(f"{'Subflow':15s}" + ''.join(f'  E={e}' for e in ease_range))
print('-' * 75)
for _, row in top4.iterrows():
    scores = [f'{row["impact"] * row["confidence"] / e:6.0f}' for e in ease_range]
    print(f"{row['subflow']:15s}" + '  '.join(scores))

print()
print("Conclusao: shopping_cart eh #1 para qualquer Ease <= 5.")
print("           So perde para credit_card se Ease(sc) >= 6 — improvavel.")

## 4. Decomposição e Decisão Final

In [ ]:
sc = sf[sf['subflow'] == 'shopping_cart'].iloc[0]
cc = sf[sf['subflow'] == 'credit_card'].iloc[0]

print("=== shopping_cart — Decomposicao do ICE ===")
print(f"  Volume (n):           {sc['n']:.0f} conversas")
print(f"  Taxa any_pattern:     {sc['any_pattern']:.1f}%")
print(f"  Impact:               {sc['impact']:.0f}")
print(f"  Confidence:           {sc['confidence']:.1f}% (P1 try-again domina quase 100%)")
print(f"  Ease:                 {sc['ease']:.0f} / 10 (adicionar 1 pergunta diagnostica)")
print(f"  ICE Score:            {sc['ice_score']:.0f}")
print(f"  ICE Normalizado:      {sc['ice_norm']:.0f} / 100")
print()
gap = (sc['ice_score'] / cc['ice_score'] - 1) * 100
print(f"Gap para o #2 (credit_card): +{gap:.0f}% acima")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

candidates = sf.nlargest(4, 'ice_score')
comp_labels = candidates['subflow'].values
highlight   = ['#d62728' if s == 'shopping_cart' else '#aec7e8' for s in comp_labels]

for ax, col, title in zip(
    axes,
    ['impact', 'confidence', 'ice_norm'],
    ['Impact (n x taxa padrao)', 'Confidence (% dominancia)', 'ICE Score (normalizado)']
):
    bars = ax.bar(comp_labels, candidates[col].values, color=highlight, edgecolor='none')
    for bar, val in zip(bars, candidates[col].values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(comp_labels, rotation=15, ha='right', fontsize=8)
    ax.set_ylim(0, candidates[col].max() * 1.25)

plt.suptitle('Comparacao Top 4 — Componentes do ICE', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_ice_decomposition_top4.png', bbox_inches='tight')
plt.show()

## 5. Sumario do Notebook 05

### ICE Ranking Final

| Rank | Subflow | Impact | Confidence | Ease | ICE | Normalizado |
|---|---|---|---|---|---|---|
| **#1** | **shopping_cart** | 213 | 99% | 2 | **10.559** | **100** |
| #2 | credit_card | 217 | 96% | 3 | 6.969 | 66 |
| #3 | slow_speed | 184 | 94% | 5 | 3.441 | 33 |
| #4 | search_results | 105 | 81% | 6 | 1.416 | 13 |

### Por que shopping_cart?

- **Impact 213:** 84,9% das 251 conversas tem padrao detectavel — maior taxa do dataset
- **Confidence 99%:** praticamente um unico padrao (P1: try-again), sem ambiguidade
- **Ease 2/10:** a intervencao e adicionar **1 pergunta diagnostica** no turno de abertura
- **Robusto:** #1 sob qualquer Ease <= 5 na analise de sensibilidade

### Backlog pos-experimento

Se o A/B de `shopping_cart` for positivo:
1. `credit_card` — intervencao similar, Ease=3
2. `slow_speed` — fluxo diagnostico multi-causa, Ease=5
3. `search_results` — requer analise adicional de causa

**Proximo passo:** `experiments/ab_test_design.md` — experiment brief com calculos de poder e regra de decisao.